# Merge History + LSLR + Inventory

This notebook merges three cleaned CSV files that are stored in the same folder as this notebook.

Expected inputs:
- reviewed history file with resolved subsystem ids
- cleaned LSLR grouped file
- cleaned inventory file

Final merge shape:
- keep history rows as the main grain
- preserve both `base_pwsid` and `resolved_pwsid`
- keep `system_name` as the raw history name
- create `display_name` for UI display, using inventory `supply_name` first and falling back to `system_name`
- merge LSLR on `base_pwsid + year`
- merge inventory on `base_pwsid`
- populate inventory fields only when `history.year == inventory_year`


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

BASE_DIR = Path('.').resolve()
OUTPUT_DIR = BASE_DIR / 'merge_output'
OUTPUT_DIR.mkdir(exist_ok=True)

history_file_name = input(
    'Enter the history CSV file name in this folder: '
).strip()
lslr_file_name = input(
    'Enter the LSLR grouped CSV file name in this folder: '
).strip()
inventory_file_name = input(
    'Enter the inventory clean CSV file name in this folder: '
).strip()
inventory_year = input(
    'Enter the inventory year (example: 2025): '
).strip()

if not history_file_name or not lslr_file_name or not inventory_file_name or not inventory_year:
    raise ValueError('You must enter all required inputs.')

inventory_year = int(inventory_year)

HISTORY_FILE_PATH = BASE_DIR / history_file_name
LSLR_FILE_PATH = BASE_DIR / lslr_file_name
INVENTORY_FILE_PATH = BASE_DIR / inventory_file_name

for path in [HISTORY_FILE_PATH, LSLR_FILE_PATH, INVENTORY_FILE_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')

print('History file:', HISTORY_FILE_PATH)
print('LSLR file:', LSLR_FILE_PATH)
print('Inventory file:', INVENTORY_FILE_PATH)
print('Inventory year:', inventory_year)
print('Output folder:', OUTPUT_DIR)


In [ ]:
history_df = pd.read_csv(HISTORY_FILE_PATH)
lslr_df = pd.read_csv(LSLR_FILE_PATH)
inventory_df = pd.read_csv(INVENTORY_FILE_PATH)

print('History rows:', len(history_df))
print('LSLR rows:', len(lslr_df))
print('Inventory rows:', len(inventory_df))

display(history_df.head())
display(lslr_df.head())
display(inventory_df.head())


In [ ]:
history_work = history_df.copy()
lslr_work = lslr_df.copy()
inventory_work = inventory_df.copy()

history_work = history_work.rename(columns={'pwsid': 'resolved_pwsid'})
history_work['resolved_pwsid'] = history_work['resolved_pwsid'].astype(str).str.strip()
history_work['base_pwsid'] = history_work['resolved_pwsid'].str.replace(r'-[A-Za-z0-9]+$', '', regex=True)
history_work['year'] = pd.to_numeric(history_work['year'], errors='coerce').astype('Int64')

lslr_work = lslr_work.rename(columns={'pwsid': 'base_pwsid'})
lslr_work['base_pwsid'] = lslr_work['base_pwsid'].astype(str).str.strip()
lslr_work['year'] = pd.to_numeric(lslr_work['year'], errors='coerce').astype('Int64')

inventory_work = inventory_work.rename(columns={'pwsid': 'base_pwsid'})
inventory_work['base_pwsid'] = inventory_work['base_pwsid'].astype(str).str.strip()
inventory_work['inventory_year'] = inventory_year

if 'inventory_missing_any' in inventory_work.columns and 'total_mismatch' in inventory_work.columns:
    inventory_work['inventory_complete_flag'] = ~(
        inventory_work['inventory_missing_any'].fillna(False).astype(bool)
        | inventory_work['total_mismatch'].fillna(False).astype(bool)
    )
else:
    inventory_work['inventory_complete_flag'] = pd.NA

inventory_work = inventory_work.rename(columns={
    'lead_lines': 'inventory_lead',
    'gpcl_lines': 'inventory_gpcl',
    'unknown_lines': 'inventory_unknown',
})

inventory_columns = [
    'base_pwsid',
    'supply_name',
    'inventory_year',
    'inventory_lead',
    'inventory_gpcl',
    'inventory_unknown',
    'total_to_identify_or_replace',
    'inventory_complete_flag',
]
inventory_columns = [col for col in inventory_columns if col in inventory_work.columns]
inventory_work = inventory_work[inventory_columns].copy()

display(history_work.head())
display(lslr_work.head())
display(inventory_work.head())


In [ ]:
merged_df = history_work.merge(
    lslr_work[['base_pwsid', 'year', 'lines_replaced']],
    on=['base_pwsid', 'year'],
    how='left'
)

merged_df = merged_df.merge(
    inventory_work,
    on='base_pwsid',
    how='left'
)

inventory_like_columns = [
    'supply_name',
    'inventory_lead',
    'inventory_gpcl',
    'inventory_unknown',
    'total_to_identify_or_replace',
    'inventory_complete_flag',
]
inventory_like_columns = [col for col in inventory_like_columns if col in merged_df.columns]

non_matching_inventory_year_mask = merged_df['year'] != merged_df['inventory_year']
for col in inventory_like_columns:
    merged_df.loc[non_matching_inventory_year_mask, col] = pd.NA

if 'supply_name' in merged_df.columns:
    merged_df['display_name'] = merged_df['supply_name'].fillna(merged_df.get('system_name'))
elif 'system_name' in merged_df.columns:
    merged_df['display_name'] = merged_df['system_name']
else:
    merged_df['display_name'] = pd.NA

desired_columns = [
    'base_pwsid',
    'resolved_pwsid',
    'system_name',
    'display_name',
    'county',
    'monitoring_end_date',
    'year',
    'lead_90th_ppb',
    'above_action_level',
    'lines_replaced',
    'inventory_year',
    'inventory_lead',
    'inventory_gpcl',
    'inventory_unknown',
    'total_to_identify_or_replace',
    'inventory_complete_flag',
]

existing_desired_columns = [col for col in desired_columns if col in merged_df.columns]
merged_df = merged_df[existing_desired_columns].copy()

merged_df = merged_df.sort_values(
    ['base_pwsid', 'year', 'monitoring_end_date', 'resolved_pwsid'],
    kind='stable'
).reset_index(drop=True)

summary_df = pd.DataFrame([
    {
        'history_rows': len(history_work),
        'lslr_rows': len(lslr_work),
        'inventory_rows': len(inventory_work),
        'merged_rows': len(merged_df),
        'exact_duplicate_rows_after_merge': int(merged_df.duplicated().sum()),
    }
])

display(summary_df)
display(merged_df.head(10))


In [ ]:
duplicate_rows = merged_df[merged_df.duplicated(keep=False)].copy()

output_stem = 'merged_history_lslr_inventory'
merged_output_path = OUTPUT_DIR / f'{output_stem}.csv'
duplicate_output_path = OUTPUT_DIR / f'{output_stem}_exact_duplicates.csv'
summary_output_path = OUTPUT_DIR / f'{output_stem}_summary.csv'

merged_df.to_csv(merged_output_path, index=False, date_format='%Y-%m-%d')
duplicate_rows.to_csv(duplicate_output_path, index=False, date_format='%Y-%m-%d')
summary_df.to_csv(summary_output_path, index=False)

print('Saved:', merged_output_path)
print('Saved:', duplicate_output_path)
print('Saved:', summary_output_path)
